# Transformer with a thermal model in PathSim (`NetlistCircuit`)

> **Note:** the component values used in this notebook are **not representative of a real transformer**; they are chosen for illustration only (visible losses, short simulation time).

1. **Part 1** - a two-winding transformer, simulated with `pathsim_rf.NetlistCircuit`.
2. **Part 2** - a second-order thermal model (3 R, 2 C), also a `NetlistCircuit`, heated by the winding losses.

In [ ]:
from pathsim import Connection, Simulation
from pathsim.blocks import Function, Scope, SinusoidalSource
from pathsim.solvers import RK4
from pathsim_rf import NetlistCircuit

## Part 1 - the transformer

```
    in   ┌───────────────┐   a                            b    ┌───────────────┐    c
     ┌───┤ R1 = 0.1 ohm  ├───●                            ●────┤ R2 = 0.5 ohm  ├────●
     │   └───────────────┘   │          k = 0.9           │    └───────────────┘    │
 ╭───┴───╮              ┌────┴────┐     ◄─ K1 ─►     ┌────┴────┐               ┌────┴────┐
(   V1    )             │  L1  ●  │                  │  L2  ●  │               │   R3    │
( ~ 100 V )             │  60 mH  │                  │  30 mH  │               │ 50 ohm  │
(  50 Hz  )             └─────────┘                  └─────────┘               └─────────┘
 ╰───┬───╯                   │                            │                         │
     │                       │                            │                         │
     └───────────────────────┴────────────────────────────┴─────────────────────────┘ GND (0)
```

`V1` is a 50 Hz sine source, `R1` / `R2` are the winding resistances, `L1` / `L2` the coupled inductances (`K1`, coefficient 0.9; `●` = polarity dot, at the top of both inductors) and `R3` the load. `GND (0)` is the reference node.

The schematic is written as a SPICE netlist, one line per element:

| Netlist line | Meaning |
|---|---|
| `V1 in 0` | voltage source between node `in` and ground `0` |
| `R1 in a {R1}` | primary winding resistance |
| `L1 a 0 {L1}` / `L2 b 0 {L2}` | the two inductors (dot = first node) |
| `K1 L1 L2 0.9` | coupling coefficient between `L1` and `L2` |
| `R2 b c {R2}`, `R3 c 0 {Rload}` | secondary winding resistance and load |

How it plugs into PathSim:

- `NetlistCircuit` is a normal block. Its **inputs** are the sources of the netlist (here `V1`), driven by a `SinusoidalSource`; its **outputs** are the signals asked for with `output_voltages` / `output_currents`.
- Only the used outputs are requested. Exporting every node and current can create direct feedthrough, which PathSim treats as an algebraic loop and makes the run much slower (see the *Performance* section of `3phase_pathsim_walkthrough.ipynb`).
- The simulation covers 3 cycles of the 50 Hz source (60 ms).

In [ ]:
FREQ = 50.0 # Hz

def transformer_netlist(R1, L1, L2, k, R2, Rload):
    return f'''
.param R1 = {R1!r}
.param L1 = {L1!r}
.param L2 = {L2!r}
.param R2 = {R2!r}
.param Rload = {Rload!r}
V1 in 0 DC 0
R1 in a {{R1}}
L1 a 0 {{L1}}
L2 b 0 {{L2}}
K1 L1 L2 {k!r}
R2 b c {{R2}}
R3 c 0 {{Rload}}
'''

def make_transformer(params):
    return NetlistCircuit(transformer_netlist(**params),
                          output_voltages=["c"],              # load voltage
                          output_currents=["R1", "R2", "R3"]) # primary, secondary and load currents

PARAMS = dict(R1=0.1, L1=60e-3, L2=30e-3, k=0.9, R2=0.5, Rload=50.0)   # ohm, H, H, - , ohm, ohm
AMPLITUDE = 100.0                                                      # V

source = SinusoidalSource(frequency=FREQ, amplitude=AMPLITUDE)
transformer = make_transformer(PARAMS)
scope = Scope(labels=["load current [A]", "primary current [A]"])
blocks = [source, transformer, scope]
connections =[Connection(source, transformer["V1"]),
              Connection(transformer["I(R3)"], scope[0]),
              Connection(transformer["I(R1)"], scope[1])]
sim = Simulation(blocks, connections, dt=1e-4, Solver=RK4, log=False)
sim.run(0.06, adaptive=False)       # 3 cycles
scope.plot()

## Part 2 - add the thermal model

The thermal network is the electrical analogy, written as a second `NetlistCircuit`: temperature rise above ambient = node voltage, heat flow = current, thermal resistance = resistor, heat capacity = capacitor, ambient = ground.

```
             w (winding)                         ┌───────────────┐   c (core)
       ┌───────────●───────────●───────────●─────┤ Rwc = 7.0 K/W ├───────────●───────────┐
       ▲           ▲           │           │     └───────────────┘           │           │
   ╭───┴───╮   ╭───┴───╮  ─────┴───── ┌────┴────┐                       ─────┴───── ┌────┴────┐
  (   Ip1   ) (   Ip2   )     Cw      │   Rwa   │                           Cc      │   Rca   │
  (  = P1   ) (  = P2   )   0.2 J/K   │ 40 K/W  │                         0.8 J/K   │ 5.5 K/W │
   ╰───────╯   ╰───────╯  ─────┬───── └─────────┘                       ─────┬───── └─────────┘
       │           │           │           │                                 │           │
       │           │           │           │                                 │           │
       │           │           │           │                                 │           │
       └───────────┴───────────┴───────────┴─────────────────────────────────┴───────────┘ ambient (0 K)
```

- `w` = windings, `c` = core / housing; `Cw`, `Cc` make it **second order**; `Rwc`, `Rca`, `Rwa` are the 3 thermal resistances.
- `Ip1` and `Ip2` are current sources (heat flow in W) injecting the winding losses `P1 = R1 i1^2` and `P2 = R2 i2^2` into the winding node `w`. They are computed by `Function` blocks from the currents `I(R1)` and `I(R2)` of the transformer block.

**Small heat capacities.** A real transformer heats up over minutes to hours, whereas the electrical side needs 20 ms steps per cycle. To keep the run short, `Cw` and `Cc` are about **1000 times smaller** than realistic values (about 200 and 800 J/K): the thermal time constants are about **1 s and 5 s** instead of about 17 min and 83 min. The steady-state temperatures depend only on the thermal resistances and the losses, so they are **not** affected; only the thermal transient is compressed (20 s simulated ~ 5.6 h real).

In [ ]:

# thermal network: K/W and J/K
THERMAL = dict(Rwc=7.0, Rca=5.5, Rwa=40.0, Cw=0.2, Cc=0.8)

THERMAL_NETLIST = f'''
.param Rwc = {THERMAL["Rwc"]}
.param Rca = {THERMAL["Rca"]}
.param Rwa = {THERMAL["Rwa"]}
.param Cw = {THERMAL["Cw"]}
.param Cc = {THERMAL["Cc"]}
Ip1 0 w 0
Ip2 0 w 0
Cw w 0 {{Cw}}
Cc c 0 {{Cc}}
Rwc w c {{Rwc}}
Rca c 0 {{Rca}}
Rwa w 0 {{Rwa}}
'''

source = SinusoidalSource(frequency=FREQ, amplitude=AMPLITUDE)
transformer = make_transformer(PARAMS)
thermal = NetlistCircuit(THERMAL_NETLIST, output_voltages=["w", "c"], output_currents=[])
# Loss could also be computed using U*I, or in this case, R*I²
loss_primary = Function(lambda i: PARAMS["R1"] * i * i)       # P1 = R1 i1^2
loss_secondary = Function(lambda i: PARAMS["R2"] * i * i)     # P2 = R2 i2^2

scope_current = Scope(labels=["load current [A]"], t_wait=19.94)   # record only the last 3 cycles
scope_temp = Scope(labels=["winding rise [K]", "core rise [K]"])
blocks_th = [source, transformer, thermal, loss_primary, loss_secondary, scope_current, scope_temp]
connections_th = [Connection(source, transformer["V1"]),
                  Connection(transformer["I(R1)"], loss_primary),
                  Connection(transformer["I(R2)"], loss_secondary),
                  Connection(transformer["I(R3)"], scope_current),
                  Connection(loss_primary, thermal["Ip1"]),
                  Connection(loss_secondary, thermal["Ip2"]),
                  Connection(thermal["V(w)"], scope_temp[0]),
                  Connection(thermal["V(c)"], scope_temp[1])]

sim = Simulation(blocks_th, connections_th, dt=1e-4, Solver=RK4, log=False)
sim.run(20.0, adaptive=False)                     # 20 s = 1000 cycles

scope_current.plot()
scope_temp.plot()